In [2]:
import os 
import numpy as np
import matplotlib.pyplot as plt
from utils.utils import *

from utils.split_condition import split_conditions
from nsddatapaper_rsa.utils.nsd_get_data import get_conditions, get_betas

In [3]:
def rescale(train, test):
    """
    Input 
    -------
    train: 1D array of length n, to rescale
    test : 1D array of length n, target to rescale to

    Output 
    --------
    new_train: 1D array of length n, rescale train array

    Rescale the train array to the test array. Linear rescale using pseudoinverse
    """
    if len(train.shape) == 1:
        train = np.array([train]) # add a dimension for concat
   # print(train.shape)
    train_ones = np.concatenate((train, np.ones((train.shape[0], train.shape[1])))).T
    try:
        scale = np.linalg.pinv(train_ones) @ test.T 
    except LinAlgError:
        scale = train_ones
        new_train = train # dont rescale
        return new_train.T.squeeze(), scale

    new_train = train_ones @ scale  #make 
    return new_train.T.squeeze(), scale

In [5]:
subj_list = ['subj01']
mode = 'train'
from numpy.linalg import LinAlgError
for i, subj in enumerate(subj_list):
        print(f'GETTING BETAS FOR {subj}')
        betas = [] 
        betas_tests = []
        tt_masks = []
        n_betas, n_voxels=0, 0
        for j in range(0, 5): 
            print(j)
        #    fit_file_check = os.path.join(fits_dir, 'fits_inversed', subj, f'fits_{subj}_train_full_{j}_raw.npy')
        #    if os.path.exists(fit_file_check):
        #        print(f'Something exists for {subj} at chunk {j}, so we assume this was computed already, and we are skiping')
         #       continue
            if j == 4:
                betas_file = os.path.join(nsd_dir, 'full_brain', subj  , f'{subj}_betas_list_{targetspace}_{mode}_full_{j}_fix.npy') # could parametertize the targetsurface but eh
            else:
                betas_file = os.path.join(nsd_dir, 'full_brain', subj  , f'{subj}_betas_list_{targetspace}_{mode}_full_{j}.npy')
            betas_chunk = np.load(betas_file, allow_pickle=True, mmap_mode='r').astype(np.float32)
                
         #   n_betas = betas_chunk.shape[0]
         #   n_voxels += betas_chunk.shape[1]
            fit_file_check = os.path.join(fits_dir, 'fits_inversed', subj, f'fits_{subj}_train_full_{j}_raw.npy')
            print(f'betas shape:  {betas_chunk.shape}')
            
            if np.isnan(betas_chunk).any():
                print(f'FOUND NANS in subj0{i+1} s betas')

            if mode == "train":
                if j == 4:
                    betas_test_file = os.path.join(nsd_dir, 'full_brain', subj  , f'{subj}_betas_list_{targetspace}_test_full_{j}_fix.npy')
                else:
                    betas_test_file = os.path.join(nsd_dir, 'full_brain', subj  , f'{subj}_betas_list_{targetspace}_test_full_{j}.npy')
                print(f'LOADING TEST BETAS FOR SUBJ0{i+1}')
                betas_test = np.load(betas_test_file, allow_pickle=True, mmap_mode='r').astype(np.float32)
                
                train_test_mask_file = os.path.join(nsd_dir, 'full_brain', subj, f'{subj}_betas_list_{targetspace}_train_test_mask_full_{j}.npy')
                print(f'LOADING TRAIN TEST MSAK FOR SUBJ0{i+1}')
                train_test_mask = np.load(train_test_mask_file, allow_pickle=True, mmap_mode='r').astype(bool)

            noise_ceilling_all_vox = np.zeros(betas_chunk.shape[0]) 
            noise_ceilling_file = os.path.join(noise_dir, f'{subj}_noise_ceilling_fullbrain_{j}.npy')
        #    if os.path.exists(noise_ceilling_file):
          #      continue


            for row in range(betas_chunk.shape[0]):  
                if row % 1000 == 0: 
                    print(f'------- FINDING NOISE CEILLING FOR ROW {row}') 
         
                current_row = np.array([betas_chunk[row]])
        

                new_row, _ = rescale(current_row, betas_test[row])
         
                rss = np.sum((new_row - betas_test[row])**2)    

                variance = np.sum((betas_test[row] - np.mean(betas_test[row]))**2)
                ve_voxels = 1 - rss / variance    # noise ceilling of all voxels in current roi 
                if row % 1000 == 0:
                    print(f'-------  NOISE CEILLING FOR ROW {row} is {ve_voxels}') 
                noise_ceilling_all_vox[row] = ve_voxels

            np.save(noise_ceilling_file, noise_ceilling_all_vox)

GETTING BETAS FOR subj01
0
betas shape:  (90725, 10000)
LOADING TEST BETAS FOR SUBJ01
LOADING TRAIN TEST MSAK FOR SUBJ01
------- FINDING NOISE CEILLING FOR ROW 0
-------  NOISE CEILLING FOR ROW 0 is 0.06755107086833001
------- FINDING NOISE CEILLING FOR ROW 1000
-------  NOISE CEILLING FOR ROW 1000 is 0.231279099552556
------- FINDING NOISE CEILLING FOR ROW 2000
-------  NOISE CEILLING FOR ROW 2000 is 0.06494245355805461
------- FINDING NOISE CEILLING FOR ROW 3000
-------  NOISE CEILLING FOR ROW 3000 is 0.09803862035974575
------- FINDING NOISE CEILLING FOR ROW 4000
-------  NOISE CEILLING FOR ROW 4000 is 0.079352463743806
------- FINDING NOISE CEILLING FOR ROW 5000
-------  NOISE CEILLING FOR ROW 5000 is 0.13123382090155522
------- FINDING NOISE CEILLING FOR ROW 6000
-------  NOISE CEILLING FOR ROW 6000 is 0.0011407609327049872
------- FINDING NOISE CEILLING FOR ROW 7000
-------  NOISE CEILLING FOR ROW 7000 is 0.0007426243969093083
------- FINDING NOISE CEILLING FOR ROW 8000
-------  

In [38]:
betas_test.shape

(90724, 10000)

In [39]:
betas_chunk.shape

(90724, 10000)

In [14]:
noise_ceilling_all_vox = np.zeros(betas_chunk.shape[0]) 
noise_ceilling_file = os.path.join(noise_dir, f'{subj}_noise_ceilling_fullbrain.npy')
if os.path.exists(noise_ceilling_file):
    continue


    for row in range(betas_chuck.shape[0]):  
            if row % 1000 == 0: 
                print(f'------- FINDING NOISE CEILLING FOR ROW {row}') 
         
            current_row = np.array([betas_train[row]])
        

            new_row, _ = rescale(current_row, betas_test[row])
   
            if row == 0:
                print(new_row)
         
            rss = np.sum((new_row - betas_test[row])**2)   
    

            variance = np.sum((betas_test[row] - np.mean(betas_test[row]))**2)
            ve_voxels = 1 - rss / variance    # noise ceilling of all voxels in current roi 
            if row % 1000 == 0:
                print(f'-------  NOISE CEILLING FOR ROW {row} is {ve_voxels}') 
            noise_ceilling_all_vox[row] = ve_voxels

        np.save(noise_ceilling_file, noise_ceilling_all_vox)